# Validating Topic Models: BERTopic + BBC News
### Basic testing notebook

Working draft assembled from the tutorial sections for hands-on testing. Run top to bottom.
Dataset: BBC News (Greene & Cunningham, 2006), loaded directly via Hugging Face `datasets`.


## 2. Dataset: BBC News (2004–2005)

2,225 full news articles, 5 categories (business, entertainment, politics, sport, tech). Loads directly -- no manual download, no encoding issues.

In [ ]:
from datasets import load_dataset, concatenate_datasets

bbc = load_dataset("SetFit/bbc-news")
bbc_all = concatenate_datasets([bbc["train"], bbc["test"]])
df = bbc_all.to_pandas()
print(df.shape)
df.head()

## 3. Preprocessing: Clean the Data

Full articles (501-25,500 characters) -- much longer than typical quasi-sentence data.
`all-MiniLM-L6-v2` truncates input past ~256 tokens (~1,000-1,500 characters), so long
articles get embedded from their first few paragraphs only. This draft proceeds with full
articles as-is, truncation included, and treats that as a documented decision (see Exercise 3).


In [ ]:
df = df.drop_duplicates(subset="text")
df = df[df["text"].str.strip().str.len() > 0].copy()
df["text"] = df["text"].str.strip()

print(f"{len(df):,} usable articles after cleaning")
df[["label", "label_text"]].value_counts()

In [ ]:
# Documents for BERTopic, plus the metadata we keep for later validation (Section 6)
documents = df["text"].tolist()
metadata = df[["label", "label_text"]].reset_index(drop=True)

## 4. Fitting the Topic Model: Embed and Sweep Candidate Models

Two hyperparameters swept, everything else fixed:

- **`min_topic_size`**: bounds 10-100 (scaled down from a CAP-sized sweep -- with only
  ~2,225 articles and 5 known categories, 200 would leave almost no topics at all)
- **`n_neighbors`** (UMAP): bounds 10 and 30, local vs. global structure


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

In [ ]:
from umap import UMAP
from bertopic import BERTopic
import pandas as pd

# Hyperparameter grid -- see markdown above for why these two parameters and these bounds
min_topic_sizes = [10, 20, 30, 50, 75, 100]
n_neighbors_values = [10, 30]

candidate_models = {}

for n_neighbors in n_neighbors_values:
    for min_topic_size in min_topic_sizes:
        umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=5,
            min_dist=0.0,
            metric="cosine",
            random_state=42,   # fixed -- only the swept parameters should vary between models
        )
        model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            min_topic_size=min_topic_size,
            calculate_probabilities=False,
            verbose=False,
        )
        topics, _ = model.fit_transform(documents, embeddings)

        n_topics = (model.get_topic_info()["Topic"] != -1).sum()
        outlier_share = (pd.Series(topics) == -1).mean()

        key = (n_neighbors, min_topic_size)
        candidate_models[key] = model
        print(f"n_neighbors={n_neighbors:>2} | min_topic_size={min_topic_size:>3} "
              f"-> {n_topics:>3} topics, {outlier_share:.1%} outliers")

## 5. Model Selection: Screening the Candidate Grid

Three automated measures, deliberately not just one: **NPMI**, **C_v** coherence
(Röder, Both & Hinneburg, 2015), and **topic diversity** (proportion of unique top-words
across topics). Two coherence metrics, not one -- Hoyle et al. (2021) showed coherence
metrics can disagree with each other and with human judgment.


In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

tokenized_docs = [doc.split() for doc in documents]
dictionary = Dictionary(tokenized_docs)

def topic_words(model, top_n=10):
    return [[word for word, _ in model.get_topic(t)][:top_n]
            for t in sorted(model.get_topics().keys()) if t != -1]

def topic_diversity(top_words):
    """Proportion of unique words across all topics' top words."""
    all_words = [w for topic in top_words for w in topic]
    return len(set(all_words)) / len(all_words) if all_words else 0.0

results = []
for (n_neighbors, min_topic_size), model in candidate_models.items():
    words = topic_words(model)
    if not words:
        continue

    npmi = CoherenceModel(topics=words, texts=tokenized_docs, dictionary=dictionary,
                           coherence="c_npmi").get_coherence()
    c_v = CoherenceModel(topics=words, texts=tokenized_docs, dictionary=dictionary,
                          coherence="c_v").get_coherence()

    results.append({
        "n_neighbors": n_neighbors,
        "min_topic_size": min_topic_size,
        "n_topics": len(words),
        "npmi": npmi,
        "c_v": c_v,
        "topic_diversity": topic_diversity(words),
    })

screening = pd.DataFrame(results).sort_values("min_topic_size")
screening

In [ ]:
# A coarse filter, not a verdict: keep models that rank in the top half on ALL THREE
# metrics at once, then read their actual topics before deciding anything.
top_half = len(screening) // 2

shortlist = screening[
    (screening["npmi"].rank(ascending=False) <= top_half) &
    (screening["c_v"].rank(ascending=False) <= top_half) &
    (screening["topic_diversity"].rank(ascending=False) <= top_half)
]
shortlist

## 6. Validating the Topic Model

Select the top of the shortlist, then run word intrusion, topic labeling, and the external comparison against BBC's existing category labels.

In [ ]:
top_two = shortlist.assign(
    avg_rank=lambda d: d[["npmi", "c_v", "topic_diversity"]].rank(ascending=False).mean(axis=1)
).nsmallest(2, "avg_rank")

selected_keys = list(zip(top_two["n_neighbors"], top_two["min_topic_size"]))
selected_models = {k: candidate_models[k] for k in selected_keys}
top_two

**Internal/semantic validity — word intrusion (Chang et al., 2009)**

In [ ]:
import random
random.seed(42)

def build_intrusion_task(model, n_top=9, n_topics_to_test=10):
    topic_ids = [t for t in sorted(model.get_topics().keys()) if t != -1]
    sampled = random.sample(topic_ids, min(n_topics_to_test, len(topic_ids)))

    tasks = []
    for t in sampled:
        top_words = [w for w, _ in model.get_topic(t)][:n_top]
        other = random.choice([o for o in topic_ids if o != t])
        intruder = random.choice([w for w, _ in model.get_topic(other)][:10])
        words = top_words + [intruder]
        random.shuffle(words)
        tasks.append({"topic": t, "words": words, "intruder": intruder})
    return tasks

# Give annotators only 'words' (never 'intruder'). Use 3 independent annotators per model.
intrusion_tasks = {key: build_intrusion_task(m) for key, m in selected_models.items()}

**Internal/content validity — topic labeling by reading documents (Maier et al., 2018)**

In [ ]:
def sample_topic_documents(model, documents, topic_id, n=10):
    idx = [i for i, t in enumerate(model.topics_) if t == topic_id]
    idx = random.sample(idx, min(n, len(idx)))
    return [documents[i] for i in idx]

for doc in sample_topic_documents(selected_models[selected_keys[0]], documents, topic_id=3):
    print("-", doc[:200], "...")

**External validity — comparison against BBC's existing category labels**

In [ ]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

def external_validity(model, metadata):
    comparison = metadata.copy()
    comparison["bertopic_topic"] = model.topics_
    comparison = comparison[comparison["bertopic_topic"] != -1]  # drop BERTopic's outlier cluster

    nmi = normalized_mutual_info_score(comparison["label"], comparison["bertopic_topic"])
    ari = adjusted_rand_score(comparison["label"], comparison["bertopic_topic"])
    return nmi, ari, comparison

for key, model in selected_models.items():
    nmi, ari, comparison = external_validity(model, metadata)
    print(f"{key}: NMI={nmi:.3f}, ARI={ari:.3f}")

In [ ]:
# Face-validity read: which BBC category dominates each BERTopic topic?
dominant_category = (
    comparison.groupby("bertopic_topic")["label_text"]
    .agg(lambda s: s.value_counts().idxmax())
)
dominant_category.head(10)

**Combine everything and check for disagreement (Section 6.4)**

In [ ]:
validation_summary = top_two.set_index(["n_neighbors", "min_topic_size"])[["npmi", "c_v", "topic_diversity"]]
# Add word-intrusion accuracy and NMI/ARI (from external_validity above) as additional
# columns once collected, then compare rankings across all five.
validation_summary

## 7. Interpreting and Labeling Topics

In [ ]:
final_key = selected_keys[0]   # set this to whichever model 6.4's comparison favored
final_model = selected_models[final_key]

topic_info = final_model.get_topic_info()
topic_info.head(15)

In [ ]:
labels = []
for topic_id in topic_info.loc[topic_info["Topic"] != -1, "Topic"]:
    top_words = [w for w, _ in final_model.get_topic(topic_id)][:10]
    sample_docs = sample_topic_documents(final_model, documents, topic_id, n=10)
    labels.append({
        "topic": topic_id,
        "top_words": top_words,
        "sample_docs": sample_docs,
        "label": None,      # fill in after reading: a short human label
        "category": None,   # interpretable / mixed / uninterpretable (Bernhard et al., 2023)
    })

labeling_worksheet = pd.DataFrame(labels)
labeling_worksheet.to_csv("topic_labeling_worksheet.csv", index=False)
labeling_worksheet.head()

Read the sample documents and fill in `label` and `category` by hand before continuing.

In [ ]:
# Example: topics 4 and 11 both turned out to be the same theme on reading -> merge them.
# final_model.merge_topics(documents, topics_to_merge=[4, 11])

# Optional: reduce toward a target count -- only after seeing what would be lost.
# final_model.reduce_topics(documents, nr_topics=15)

In [ ]:
custom_labels = {
    row["topic"]: row["label"]
    for _, row in labeling_worksheet.iterrows() if row["label"]
}
final_model.set_topic_labels(custom_labels)
final_model.get_topic_info().head(15)

## 8. Reporting & Documenting Validation

In [ ]:
report = {
    "model": {
        "method": "BERTopic",
        "embedding_model": "all-MiniLM-L6-v2",
        "umap_n_neighbors": final_key[0],
        "min_topic_size": final_key[1],
        "random_state": 42,
        "n_topics": int((topic_info["Topic"] != -1).sum()),
    },
    "model_selection_stage": {
        "candidates_considered": len(candidate_models),
        "screening_method": "NPMI + C_v coherence + topic diversity; kept models ranking "
                             "in the top half on all three",
        "npmi": float(screening.loc[
            (screening.n_neighbors == final_key[0]) & (screening.min_topic_size == final_key[1]), "npmi"
        ].iloc[0]),
        "c_v": float(screening.loc[
            (screening.n_neighbors == final_key[0]) & (screening.min_topic_size == final_key[1]), "c_v"
        ].iloc[0]),
        "topic_diversity": float(screening.loc[
            (screening.n_neighbors == final_key[0]) & (screening.min_topic_size == final_key[1]), "topic_diversity"
        ].iloc[0]),
    },
    "model_validation_stage": {
        "word_intrusion_accuracy": None,  # fill in: mean % intruders correctly identified
        "topic_labeling": {
            "interpretable_pct": (labeling_worksheet["category"] == "interpretable").mean(),
            "mixed_pct": (labeling_worksheet["category"] == "mixed").mean(),
            "uninterpretable_pct": (labeling_worksheet["category"] == "uninterpretable").mean(),
        },
        "external_validity_nmi": None,  # from external_validity() above
        "external_validity_ari": None,
    },
}

In [ ]:
def render_report(report):
    lines = ["# Validation Report\n", "## Model\n"]
    lines += [f"- **{k}**: {v}" for k, v in report["model"].items()]
    lines.append("\n## Model Selection (Section 5)\n")
    lines += [f"- **{k}**: {v}" for k, v in report["model_selection_stage"].items()]
    lines.append("\n## Model Validation (Sections 6-7)\n")
    for k, v in report["model_validation_stage"].items():
        if isinstance(v, dict):
            lines.append(f"- **{k}**:")
            lines += [f"  - {kk}: {vv}" for kk, vv in v.items()]
        else:
            lines.append(f"- **{k}**: {v}")
    return "\n".join(lines)

print(render_report(report))
with open("validation_report.md", "w") as f:
    f.write(render_report(report))